# Rebuilt Modelling Section

This notebook rebuilds the batch modelling section end to end: the section 3 scope filter, the full 4.21 entity-embedding pipeline with the per-pair model selection router, and the pooling-vs-per-pair comparison and exclusion diagnostics. It is meant to be merged into the existing capstone notebook, in place of the current section 3.3 and section 4.21 content.

It depends on objects already defined earlier in the notebook: `retail`, `unit_to_kg`, `master`, `shortlist`, `create_sequences`, and the standard imports (`pandas as pd`, `numpy as np`, `matplotlib.pyplot as plt`, `MinMaxScaler`, `mean_absolute_error`, `Sequential`, `LSTM`, `Dropout`, `Dense`, `EarlyStopping`). It does not redefine those.


## 3.3.1 Scoping Out Non-Food and Unit-Incompatible Commodities

Before completeness or reporting span is computed, two categories of commodity are removed from `retail` entirely, rather than being left to silently fail three sections later at the price-per-kg conversion step.

Fuel (diesel, kerosene, petrol-gasoline) is excluded on scope grounds: this is a food price forecasting project, and these three commodities are present in the WFP dataset because it tracks a broader commodity basket than food alone.

Commodities priced in a unit that `unit_to_kg` has no entry for, such as litres, millilitres, or a bare unit count, are excluded on measurement grounds. Milk in all four varieties, vegetable oil, and bananas fall into this group. These are legitimate food commodities, but they are priced by volume or by count, not by weight, so a per-kilogram price index was never the right fit for them. Filtering by unit compatibility, rather than hardcoding these specific names, also protects against any other commodity in the dataset sharing the same problem.


In [ ]:
FUEL_COMMODITIES = ["Fuel (diesel)", "Fuel (kerosene)", "Fuel (petrol-gasoline)"]

excluded_units = sorted(set(retail["unit"]) - set(unit_to_kg))
unit_excluded_commodities = retail[retail["unit"].isin(excluded_units)]["commodity"].unique()

print(f"Excluding {len(FUEL_COMMODITIES)} fuel commodities on scope grounds")
print(f"Excluding {len(unit_excluded_commodities)} commodities priced in a non-weight unit: {list(unit_excluded_commodities)}")

retail = retail[
    ~retail["commodity"].isin(FUEL_COMMODITIES) &
    retail["unit"].isin(unit_to_kg)
].copy()

print(f"Retail rows after scope filtering: {len(retail)}")


This must run before `reporting_span` and the `MIN_YEARS_ACTIVE` completeness floor further down in section 3. Everything from `reporting_span` through `shortlist` and `master` needs a full re-run after this filter changes.


## 4.21 Batch Modelling Across Market-Commodity Pairs


### 4.21.1 Encode Market and Commodity as Entities

Encoders are fit on `shortlist` rather than `master`, since a small number of shortlisted commodities can still be fully filtered out of `master` at the price-per-kg conversion step and never reach the modelling table. Fitting on `shortlist` ensures every pair the batch loop iterates over has a valid entity id, even if that pair is later skipped for having no usable rows.


In [ ]:
from sklearn.preprocessing import LabelEncoder

market_encoder = LabelEncoder()
commodity_encoder = LabelEncoder()

market_encoder.fit(shortlist["market"])
commodity_encoder.fit(shortlist["commodity"])

master["market_id"] = master["market"].map(lambda m: market_encoder.transform([m])[0])
master["commodity_id"] = master["commodity"].map(lambda c: commodity_encoder.transform([c])[0])

n_markets = len(market_encoder.classes_)
n_commodities = len(commodity_encoder.classes_)


`n_markets` and `n_commodities` reflect every entity in the shortlist, including any that end up contributing zero rows to the batch.


### 4.21.2 Build Per-Pair Sequences with Entity Identifiers

Each pair gets its own chronological train, validation, and test split, and its own `MinMaxScaler` fit only on that pair's training rows. The target is the scaled month-over-month price change (`price_diff`), not the price level, since a differenced series stays far more stable across time than a trending level does, and pooling levels across pairs with different trends was previously causing validation loss to be dominated by a handful of badly extrapolated pairs.

Raw, unscaled last-known-price and actual-price arrays are kept for both the validation and test splits. Validation copies let a model-selection decision get made later without ever looking at test data; test copies are used only for the final, honest performance number.


In [ ]:
FEATURES = ["price_diff", "rainfall_lag_3", "rainfall_lag_4", "temperature_lag_3", "temperature_lag_4"]
LOOKBACK = 6
MIN_ROWS = LOOKBACK + 4

def build_pair_sequences(pair_df, market_id, commodity_id, model_track, lookback=LOOKBACK):
    pair_df = pair_df.sort_values("date").copy()
    pair_df["price_diff"] = pair_df["price_per_kg"].diff()
    pair_df = pair_df.dropna(subset=FEATURES)

    if len(pair_df) < MIN_ROWS:
        return None

    train_end = pair_df["date"].quantile(0.8)
    val_end = pair_df["date"].quantile(0.9)

    train_mask = pair_df["date"] <= train_end
    val_mask = (pair_df["date"] > train_end) & (pair_df["date"] <= val_end)
    test_mask = pair_df["date"] > val_end

    naive_pred = pair_df["price_per_kg"].shift(1)
    naive_ready = test_mask & naive_pred.notna()
    if not naive_ready.any():
        return None

    naive_actual = pair_df.loc[naive_ready, "price_per_kg"]
    naive_pred_vals = naive_pred.loc[naive_ready]
    naive_mae = mean_absolute_error(naive_actual, naive_pred_vals)
    naive_mape = np.mean(np.abs((naive_actual - naive_pred_vals) / naive_actual)) * 100

    val_naive_ready = val_mask & naive_pred.notna()
    if val_naive_ready.any():
        val_naive_actual = pair_df.loc[val_naive_ready, "price_per_kg"]
        val_naive_pred_vals = naive_pred.loc[val_naive_ready]
        val_naive_mae = mean_absolute_error(val_naive_actual, val_naive_pred_vals)
    else:
        val_naive_mae = np.nan

    scaler = MinMaxScaler()
    scaler.fit(pair_df.loc[train_mask, FEATURES])

    train_scaled = scaler.transform(pair_df.loc[train_mask, FEATURES])
    val_scaled = scaler.transform(pair_df.loc[val_mask, FEATURES])
    test_scaled = scaler.transform(pair_df.loc[test_mask, FEATURES])

    if len(train_scaled) <= lookback:
        return None

    X_train, y_train = create_sequences(train_scaled, lookback)

    val_source = np.vstack([train_scaled[-lookback:], val_scaled]) if len(val_scaled) > 0 else train_scaled[-lookback:]
    X_val, y_val = create_sequences(val_source, lookback)

    test_source = np.vstack([val_scaled[-lookback:], test_scaled]) if len(val_scaled) >= lookback else np.vstack([train_scaled[-lookback:], test_scaled])
    X_test, y_test = create_sequences(test_source, lookback)

    if len(X_train) < 1 or len(X_test) == 0:
        return None

    last_price_val = pair_df["price_per_kg"].shift(1).loc[val_mask].values
    actual_price_val = pair_df.loc[val_mask, "price_per_kg"].values

    last_price_test = pair_df["price_per_kg"].shift(1).loc[test_mask].values
    actual_price_test = pair_df.loc[test_mask, "price_per_kg"].values

    return {
        "market": pair_df["market"].iloc[0],
        "commodity": pair_df["commodity"].iloc[0],
        "market_id": market_id,
        "commodity_id": commodity_id,
        "model_track": model_track,
        "scaler": scaler,
        "naive_mae": naive_mae,
        "naive_mape": naive_mape,
        "val_naive_mae": val_naive_mae,
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test,
        "last_price_val": last_price_val,
        "actual_price_val": actual_price_val,
        "last_price_test": last_price_test,
        "actual_price_test": actual_price_test,
    }


Pairs that fail the minimum row requirement, end up with no usable test sequences after the lookback window is applied, or have no naive-ready test rows, return `None` and are skipped in the next step.


### 4.21.3 Combine Sequences Into a Single Training Batch

Every pair's train, validation, and test sequences are stacked into one array, with parallel arrays of `market_id` and `commodity_id` so each row still carries its entity identity. Validation-side last-known-price and actual-price arrays are pooled alongside the test-side ones, so a per-pair validation score can be computed after training without ever touching test data.


In [ ]:
pair_bundles = []
X_train_list, y_train_list, mkt_train_list, com_train_list = [], [], [], []
X_val_list, y_val_list, mkt_val_list, com_val_list = [], [], [], []
X_test_list, y_test_list, mkt_test_list, com_test_list = [], [], [], []
pair_idx_val_list, pair_idx_test_list = [], []
last_price_val_list, actual_price_val_list = [], []
last_price_test_list, actual_price_test_list = [], []

for _, row in shortlist.iterrows():
    pair_df = master[(master["market"] == row["market"]) & (master["commodity"] == row["commodity"])]

    market_id = market_encoder.transform([row["market"]])[0]
    commodity_id = commodity_encoder.transform([row["commodity"]])[0]

    bundle = build_pair_sequences(pair_df, market_id, commodity_id, row["model_track"])
    if bundle is None:
        continue

    pair_idx = len(pair_bundles)
    pair_bundles.append(bundle)

    n_tr, n_va, n_te = len(bundle["X_train"]), len(bundle["X_val"]), len(bundle["X_test"])

    X_train_list.append(bundle["X_train"]); y_train_list.append(bundle["y_train"])
    mkt_train_list.append(np.full(n_tr, market_id)); com_train_list.append(np.full(n_tr, commodity_id))

    X_val_list.append(bundle["X_val"]); y_val_list.append(bundle["y_val"])
    mkt_val_list.append(np.full(n_va, market_id)); com_val_list.append(np.full(n_va, commodity_id))
    pair_idx_val_list.append(np.full(n_va, pair_idx))
    last_price_val_list.append(bundle["last_price_val"])
    actual_price_val_list.append(bundle["actual_price_val"])

    X_test_list.append(bundle["X_test"]); y_test_list.append(bundle["y_test"])
    mkt_test_list.append(np.full(n_te, market_id)); com_test_list.append(np.full(n_te, commodity_id))
    pair_idx_test_list.append(np.full(n_te, pair_idx))
    last_price_test_list.append(bundle["last_price_test"])
    actual_price_test_list.append(bundle["actual_price_test"])

X_train_all = np.concatenate(X_train_list)
y_train_all = np.concatenate(y_train_list)
market_train_all = np.concatenate(mkt_train_list)
commodity_train_all = np.concatenate(com_train_list)

X_val_all = np.concatenate(X_val_list)
y_val_all = np.concatenate(y_val_list)
market_val_all = np.concatenate(mkt_val_list)
commodity_val_all = np.concatenate(com_val_list)
pair_idx_val_all = np.concatenate(pair_idx_val_list)
last_price_val_all = np.concatenate(last_price_val_list)
actual_price_val_all = np.concatenate(actual_price_val_list)

X_test_all = np.concatenate(X_test_list)
y_test_all = np.concatenate(y_test_list)
market_test_all = np.concatenate(mkt_test_list)
commodity_test_all = np.concatenate(com_test_list)
pair_idx_test_all = np.concatenate(pair_idx_test_list)
last_price_test_all = np.concatenate(last_price_test_list)
actual_price_test_all = np.concatenate(actual_price_test_list)

print(f"Pairs included: {len(pair_bundles)} out of {len(shortlist)}")


The gap between "pairs included" and the total shortlist size shows how many pairs were too short even for the lookback window, or had no naive-ready test period, to contribute to the batch.


In [ ]:
print(f"y_val_all range: [{y_val_all.min():.2f}, {y_val_all.max():.2f}]")
print(f"y_test_all range: [{y_test_all.min():.2f}, {y_test_all.max():.2f}]")
print(f"Share of y_val_all outside [0, 1]: {((y_val_all < 0) | (y_val_all > 1)).mean() * 100:.1f}%")
print(f"Share of y_test_all outside [0, 1]: {((y_test_all < 0) | (y_test_all > 1)).mean() * 100:.1f}%")


Values outside `[0, 1]` mean a pair's own scaler, fit only on its training window, is being extrapolated on validation or test data whose prices moved past what training ever saw. Differencing before scaling keeps this share far smaller than it was when the model predicted absolute price levels.


### 4.21.4 Global Entity Embedding LSTM Architecture

The model takes three inputs: the numeric sequence window, the market id, and the commodity id. The sequence branch is an LSTM, kept deliberately small since the pooled training set, while large in total row count, is still built from many short and individually noisy series. The market and commodity branches are `Embedding` layers mapping each integer id to a learned dense vector, concatenated with the LSTM's output before the final dense layers.

Every learnable block carries its own regularization: `recurrent_dropout` and an L2 penalty on the LSTM's kernel, an L2 penalty on both embedding tables, and dropout before and after the merge. This targets the train/validation gap directly. An LSTM and embedding tables with no constraints have more than enough capacity to memorize which specific market or commodity a sequence belongs to rather than learning a pattern that transfers across series.


In [ ]:
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

n_features = len(FEATURES)
market_embed_dim = min(8, (n_markets + 1) // 2)
commodity_embed_dim = min(8, (n_commodities + 1) // 2)

sequence_input = Input(shape=(LOOKBACK, n_features), name="sequence_input")
market_input = Input(shape=(1,), name="market_input")
commodity_input = Input(shape=(1,), name="commodity_input")

market_embed = Embedding(
    n_markets, market_embed_dim,
    embeddings_regularizer=l2(0.01), name="market_embedding"
)(market_input)
market_embed = Flatten()(market_embed)

commodity_embed = Embedding(
    n_commodities, commodity_embed_dim,
    embeddings_regularizer=l2(0.01), name="commodity_embedding"
)(commodity_input)
commodity_embed = Flatten()(commodity_embed)

lstm_out = LSTM(
    16, return_sequences=False,
    recurrent_dropout=0.2, kernel_regularizer=l2(0.01)
)(sequence_input)
lstm_out = Dropout(0.3)(lstm_out)

merged = Concatenate()([lstm_out, market_embed, commodity_embed])
dense_out = Dense(16, activation="relu", kernel_regularizer=l2(0.01))(merged)
dense_out = Dropout(0.3)(dense_out)
output = Dense(1)(dense_out)

embedding_model = Model(
    inputs=[sequence_input, market_input, commodity_input],
    outputs=output
)
embedding_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
embedding_model.summary()


Embedding dimensions stay capped at 8, scaled to roughly half the number of unique entities. LSTM units are cut to 16, since a smaller pooled model that generalizes is more useful here than a larger one that memorizes individual series. If the train/validation gap is still wide after this, tightening dropout further or dropping LSTM units to 8 is the next lever, not adding capacity back.


### 4.21.5 Train the Shared Model Across All Pairs

Training happens once, on the pooled batch from every pair, rather than once per pair. Early stopping on validation loss restores the best weights once validation performance stops improving.


In [ ]:
early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

embedding_history = embedding_model.fit(
    [X_train_all, market_train_all, commodity_train_all],
    y_train_all,
    validation_data=([X_val_all, market_val_all, commodity_val_all], y_val_all),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(embedding_history.history["loss"], label="Training Loss")
plt.plot(embedding_history.history["val_loss"], label="Validation Loss")
plt.title("Entity Embedding LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


This plot shows a single training curve for every pair combined, rather than a separate curve per pair. A converging validation loss that tracks training loss, rather than sitting flat far above it, indicates the shared representation is generalizing across markets and commodities.


### 4.21.6 Per-Pair Model Selection

Each pair's own validation performance decides whether it gets the learned forecast or the naive baseline, so the choice is never made using test data. Two conditions must both hold before a pair is trusted with the model's forecast: at least 4 validation rows to compare against, since a win on 2 or 3 points is noise wearing the shape of a signal, and an improvement over naive of more than 15 percent, since a thin edge on a handful of points isn't a real edge. Any pair that doesn't clear both bars defaults to naive. This means the system can only ever be as good as naive, never worse, and only earns the right to do better where the evidence is actually strong enough to trust.


In [ ]:
MIN_VAL_ROWS_FOR_SELECTION = 4
MODEL_IMPROVEMENT_MARGIN = 0.15

predicted_val_delta = embedding_model.predict(
    [X_val_all, market_val_all, commodity_val_all], verbose=0
).flatten()

predicted_test_delta = embedding_model.predict(
    [X_test_all, market_test_all, commodity_test_all], verbose=0
).flatten()

selection_results = []

for pair_idx, bundle in enumerate(pair_bundles):
    val_mask_rows = pair_idx_val_all == pair_idx
    test_mask_rows = pair_idx_test_all == pair_idx

    if not test_mask_rows.any():
        continue

    scaler = bundle["scaler"]

    chosen = "naive"
    if val_mask_rows.sum() >= MIN_VAL_ROWS_FOR_SELECTION and not np.isnan(bundle["val_naive_mae"]):
        val_diff_matrix = np.zeros((val_mask_rows.sum(), n_features))
        val_diff_matrix[:, 0] = predicted_val_delta[val_mask_rows]
        val_predicted_diff = scaler.inverse_transform(val_diff_matrix)[:, 0]
        val_pred = last_price_val_all[val_mask_rows] + val_predicted_diff
        val_actual = actual_price_val_all[val_mask_rows]
        val_model_mae = mean_absolute_error(val_actual, val_pred)

        if bundle["val_naive_mae"] > 0:
            improvement = (bundle["val_naive_mae"] - val_model_mae) / bundle["val_naive_mae"]
            if improvement > MODEL_IMPROVEMENT_MARGIN:
                chosen = "model"
        elif val_model_mae == 0:
            chosen = "model"

    test_diff_matrix = np.zeros((test_mask_rows.sum(), n_features))
    test_diff_matrix[:, 0] = predicted_test_delta[test_mask_rows]
    test_predicted_diff = scaler.inverse_transform(test_diff_matrix)[:, 0]
    model_pred = last_price_test_all[test_mask_rows] + test_predicted_diff
    naive_pred_test = last_price_test_all[test_mask_rows]
    actual = actual_price_test_all[test_mask_rows]

    final_pred = model_pred if chosen == "model" else naive_pred_test
    final_mae = mean_absolute_error(actual, final_pred)
    nz = actual != 0
    final_mape = np.mean(np.abs((actual[nz] - final_pred[nz]) / actual[nz])) * 100

    selection_results.append({
        "market": bundle["market"],
        "commodity": bundle["commodity"],
        "model_track": bundle["model_track"],
        "chosen_forecast": chosen,
        "final_mae": final_mae,
        "final_mape": final_mape,
        "naive_mae": bundle["naive_mae"],
    })

selection_results = pd.DataFrame(selection_results)
selection_results["beats_naive"] = selection_results["final_mae"] <= selection_results["naive_mae"]


In [ ]:
print(f"Pairs routed to model: {(selection_results['chosen_forecast'] == 'model').sum()}")
print(f"Pairs routed to naive: {(selection_results['chosen_forecast'] == 'naive').sum()}")
print(f"Overall mean MAE with routing: {selection_results['final_mae'].mean():.2f}")
print(f"Overall mean MAPE with routing: {selection_results['final_mape'].mean():.2f}%")
print(f"Pairs at or better than naive: {selection_results['beats_naive'].mean() * 100:.1f}%")


In [ ]:
model_routed = selection_results[selection_results["chosen_forecast"] == "model"]
print(f"Model-routed pairs: {len(model_routed)}")
print(f"Still beat or tie naive on test: {model_routed['beats_naive'].sum()}")
print(f"Lost to naive on test despite winning on validation: {(~model_routed['beats_naive']).sum()}")


By construction, every pair routed to naive ties naive exactly, so the overall beat-rate above is only meaningful once checked against how many of the pairs actually routed to the model held up on the untouched test set, which is what the diagnostic above reports.


### 4.21.7 Pooling Versus Per-Pair Models on the Short-History Track

The pooled, entity-embedding model is only worth its added complexity if it actually outperforms fitting an independent model per pair, specifically on the `lstm` track, the short-history series the pooling approach was designed to help. This rebuilds the original per-pair approach, deliberately shrunk to a comparable capacity, and compares it against the pooled model's own performance on the same pairs, before the per-pair routing layer above is applied, so the two improvements aren't blurred together.


In [ ]:
WEATHER_FEATS = ["rainfall_lag_3", "rainfall_lag_4", "temperature_lag_3", "temperature_lag_4"]

def evaluate_pair(market, commodity, model_track, min_rows=10, lookback=6):
    pair = master[(master["market"] == market) & (master["commodity"] == commodity)].copy()
    pair = pair.sort_values("date").dropna(subset=["price_per_kg"] + WEATHER_FEATS)

    if len(pair) < min_rows:
        return None

    train_end = pair["date"].quantile(0.8)
    val_end = pair["date"].quantile(0.9)

    pair["naive_pred"] = pair["price_per_kg"].shift(1)
    test_naive = pair[pair["date"] > val_end].dropna(subset=["naive_pred"])
    if test_naive.empty:
        return None
    naive_mae = mean_absolute_error(test_naive["price_per_kg"], test_naive["naive_pred"])
    naive_mape = np.mean(np.abs((test_naive["price_per_kg"] - test_naive["naive_pred"])
                                 / test_naive["price_per_kg"])) * 100

    feats = ["price_per_kg"] + WEATHER_FEATS
    train_m = pair["date"] <= train_end
    val_m = (pair["date"] > train_end) & (pair["date"] <= val_end)
    test_m = pair["date"] > val_end

    scaler_p = MinMaxScaler()
    scaler_p.fit(pair.loc[train_m, feats])
    train_s = scaler_p.transform(pair.loc[train_m, feats])
    val_s = scaler_p.transform(pair.loc[val_m, feats])
    test_s = scaler_p.transform(pair.loc[test_m, feats])

    if len(train_s) <= lookback:
        return None

    X_tr, y_tr = create_sequences(train_s, lookback)
    val_source = np.vstack([train_s[-lookback:], val_s]) if len(val_s) > 0 else train_s[-lookback:]
    X_va, y_va = create_sequences(val_source, lookback)
    test_source = np.vstack([val_s[-lookback:], test_s]) if len(val_s) >= lookback else np.vstack([train_s[-lookback:], test_s])
    X_te, y_te = create_sequences(test_source, lookback)

    if len(X_tr) < 1 or len(X_te) == 0:
        return None

    try:
        model = Sequential([
            LSTM(8, input_shape=(X_tr.shape[1], X_tr.shape[2]), return_sequences=False,
                 recurrent_dropout=0.2),
            Dropout(0.3),
            Dense(1)
        ])
        model.compile(optimizer="adam", loss="mse")
        es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
        model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=40,
                  batch_size=4, callbacks=[es], verbose=0)

        pred_scaled = model.predict(X_te, verbose=0).flatten()
        pm = np.zeros((len(pred_scaled), len(feats))); pm[:, 0] = pred_scaled
        pred = scaler_p.inverse_transform(pm)[:, 0]

        am = np.zeros((len(y_te), len(feats))); am[:, 0] = y_te
        actual = scaler_p.inverse_transform(am)[:, 0]

        mae = mean_absolute_error(actual, pred)
        nz = actual != 0
        mape = np.mean(np.abs((actual[nz] - pred[nz]) / actual[nz])) * 100
    except Exception as e:
        return {"market": market, "commodity": commodity, "n_test": len(test_naive),
                "naive_mae": naive_mae, "naive_mape": naive_mape,
                "model_mae": np.nan, "model_mape": np.nan, "error": str(e)}

    return {"market": market, "commodity": commodity, "n_test": len(test_naive),
            "naive_mae": naive_mae, "naive_mape": naive_mape,
            "model_mae": mae, "model_mape": mape}


In [ ]:
lstm_shortlist = shortlist[shortlist["model_track"] == "lstm"]

per_pair_lstm_results = []
for _, row in lstm_shortlist.iterrows():
    res = evaluate_pair(row["market"], row["commodity"], row["model_track"])
    if res is not None:
        per_pair_lstm_results.append(res)

per_pair_lstm_results = pd.DataFrame(per_pair_lstm_results)
per_pair_lstm_results["beats_naive"] = per_pair_lstm_results["model_mae"] < per_pair_lstm_results["naive_mae"]


In [ ]:
pooled_only_results = []
for pair_idx, bundle in enumerate(pair_bundles):
    test_mask_rows = pair_idx_test_all == pair_idx
    if not test_mask_rows.any():
        continue
    scaler = bundle["scaler"]
    test_diff_matrix = np.zeros((test_mask_rows.sum(), n_features))
    test_diff_matrix[:, 0] = predicted_test_delta[test_mask_rows]
    test_predicted_diff = scaler.inverse_transform(test_diff_matrix)[:, 0]
    pred = last_price_test_all[test_mask_rows] + test_predicted_diff
    actual = actual_price_test_all[test_mask_rows]
    mae = mean_absolute_error(actual, pred)
    nz = actual != 0
    mape = np.mean(np.abs((actual[nz] - pred[nz]) / actual[nz])) * 100
    pooled_only_results.append({"market": bundle["market"], "commodity": bundle["commodity"],
                                 "model_track": bundle["model_track"], "pooled_mae": mae, "pooled_mape": mape})

pooled_only_results = pd.DataFrame(pooled_only_results)
by_track = pooled_only_results.groupby("model_track")[["pooled_mae", "pooled_mape"]].mean()


In [ ]:
per_pair_mae = per_pair_lstm_results["model_mae"].mean()
per_pair_mape = per_pair_lstm_results["model_mape"].mean()
per_pair_beat_rate = per_pair_lstm_results["beats_naive"].mean() * 100

pooled_lstm = by_track.loc["lstm"]

print(f"Per-pair LSTM,  lstm-track only:  MAE {per_pair_mae:.2f}, MAPE {per_pair_mape:.2f}%, beat-naive {per_pair_beat_rate:.1f}%")
print(f"Pooled embedding, lstm-track only (before routing): MAE {pooled_lstm['pooled_mae']:.2f}, MAPE {pooled_lstm['pooled_mape']:.2f}%")


### 4.21.8 Diagnosing Excluded Pairs

This checks each shortlisted pair that never made it into `pair_bundles` against the same guard clauses in `build_pair_sequences`, to find out which specific condition filtered it out, rather than treating all excluded pairs as one undifferentiated group. If the section 3.3.1 scope filter above is already in place, most of what this used to catch (litre- and count-priced commodities) should no longer appear here at all.


In [ ]:
def diagnose_pair(pair_df, lookback=LOOKBACK, min_rows=MIN_ROWS):
    pair_df = pair_df.sort_values("date").copy()
    pair_df["price_diff"] = pair_df["price_per_kg"].diff()
    before = len(pair_df)
    pair_df = pair_df.dropna(subset=FEATURES)
    after = len(pair_df)

    if after < min_rows:
        return f"too few rows after dropna ({after} rows, started at {before})"

    train_end = pair_df["date"].quantile(0.8)
    val_end = pair_df["date"].quantile(0.9)
    train_mask = pair_df["date"] <= train_end
    test_mask = pair_df["date"] > val_end

    if train_mask.sum() <= lookback:
        return f"train rows ({train_mask.sum()}) too few for lookback ({lookback})"

    naive_pred = pair_df["price_per_kg"].shift(1)
    naive_ready = test_mask & naive_pred.notna()
    if not naive_ready.any():
        return "no usable test rows for naive baseline"

    return "ok"

diagnosis = []
for _, row in shortlist.iterrows():
    pair_df = master[(master["market"] == row["market"]) & (master["commodity"] == row["commodity"])]
    reason = diagnose_pair(pair_df)
    diagnosis.append({"market": row["market"], "commodity": row["commodity"],
                       "model_track": row["model_track"], "reason": reason})

diagnosis = pd.DataFrame(diagnosis)
excluded = diagnosis[diagnosis["reason"] != "ok"]

print(excluded["reason"].value_counts())
print()
print(excluded.groupby("model_track")["reason"].value_counts())
